# IKpy Quickstart #

Import the IKPy module : 

In [49]:
import ikpy.chain
import numpy as np
import ikpy.utils.plot as plot_utils

The basic element of IKPy is the kinematic `Chain`.
To create a chain from an URDF file : 

In [50]:
# will give warnings, that's OK
my_chain = ikpy.chain.Chain.from_urdf_file("./ur5/ur5_gripper.urdf")

/local/hammits/lab-2-ik/.pixi/envs/default/lib/python3.12/site-packages/ikpy/chain.py:60: UserWarning: Link Base link (index: 0) is of type 'fixed' but set as active in the active_links_mask. In practice, this fixed link doesn't provide any transformation so is as it were inactive
  warnings.warn("Link {} (index: {}) is of type 'fixed' but set as active in the active_links_mask. In practice, this fixed link doesn't provide any transformation so is as it were inactive".format(link.name, link_index))
/local/hammits/lab-2-ik/.pixi/envs/default/lib/python3.12/site-packages/ikpy/chain.py:60: UserWarning: Link ee_fixed_joint (index: 7) is of type 'fixed' but set as active in the active_links_mask. In practice, this fixed link doesn't provide any transformation so is as it were inactive
  warnings.warn("Link {} (index: {}) is of type 'fixed' but set as active in the active_links_mask. In practice, this fixed link doesn't provide any transformation so is as it were inactive".format(link.name, 

# Inverse kinematics

In Inverse Kinematics, you want the "end effector" of your robot to reach a specific 3D position (and in general, orientation) in space. The end effector is the final link of the robot arm, furthest from the base, and possibly a tool that is attached (a gripper in our robot model).

To have a more general representation of position, IKPy works with homogeneous coordinates. Homogenous coordinates are represented as a 4x4 matrix storing both position and orientation. Here we only consider (x,y,z) position, not orientation of the chain.

In [51]:
# define target X, Y, Z coordinate
target_position = [ 1.0872, 0, 0]

In [52]:
# run inverse kinematics solver
print("The joint angles are : ", my_chain.inverse_kinematics(target_position))

The joint angles are :  [ 0.00000000e+00 -1.33227902e-01  8.25717421e-02 -4.96493013e-05
 -2.27617171e+00 -1.36756048e+00  0.00000000e+00  0.00000000e+00]


You can check that the Inverse Kinematics is correct by comparing with the original position vector : 

In [53]:
# run forward kinematics on the inverse kinematics solution to verify
real_frame = my_chain.forward_kinematics(my_chain.inverse_kinematics(target_position))
print("Computed position vector : %s, goal position vector : %s" % (real_frame[:3, 3], target_position))

Computed position vector : [ 9.46755389e-01 -2.38414298e-08  1.15181567e-02], goal position vector : [1.0872, 0, 0]


In [54]:
def sphere(n, radius):
    count = 0;
    points = [];
    while count < n:
        point = np.random.randn(3);
        length = np.linalg.norm(point);
        for x in point:
            x = (x/length)*radius;
        count += 1;
        points.append(point);
    return points;

targets = sphere(100, radius=2.0)

reached = []
for target in targets:
    joints = my_chain.inverse_kinematics(target);
    fk = my_chain.forward_kinematics (joints);
    reached.append(fk[:3, 3]);

reached = np.array(reached);
print(reached);

[[ 3.77875966e-01  1.02621416e-01 -7.76319173e-01]
 [ 2.81343457e-01  8.53629828e-01  2.66291619e-01]
 [ 7.97039519e-01 -4.57901146e-01  3.28801245e-01]
 [-2.38706631e-01 -6.31598244e-01 -5.79030095e-01]
 [-8.56069504e-01 -4.06689508e-01  1.53365162e-01]
 [ 4.81556810e-01  1.61782277e-01  7.89819026e-01]
 [-2.29444938e-01 -1.40654065e-01 -8.21854318e-01]
 [-6.54452437e-02  1.03468700e-01  5.88282356e-02]
 [-4.12172549e-02 -9.44868839e-01  2.89725165e-04]
 [ 1.06768236e-01 -2.15015047e-01 -8.29939979e-01]
 [ 6.21128446e-01  4.63699278e-01  4.86789391e-01]
 [ 4.19714944e-01 -4.33774189e-01 -6.44362089e-01]
 [-2.29646514e-01 -6.71407092e-01  7.20703822e-01]
 [-2.52169834e-01  8.63931301e-01 -2.14826093e-01]
 [ 5.90764076e-01  5.07688685e-01  6.32872032e-01]
 [ 5.95338715e-01  4.92530161e-01  6.41752632e-01]
 [ 7.44783683e-01 -3.94637156e-01 -3.48942186e-01]
 [ 7.29958891e-02 -8.67351847e-01 -1.96474271e-01]
 [ 3.81648941e-01  7.54060816e-01 -3.44557771e-01]
 [ 4.94187868e-01  7.57363706e-

# Plotting
And finally plot the result : 

In [59]:
from ikpy.utils import geometry
def get_joints():
    nodes = []
    # "chain" is the Chain object returned by ikpy when it loads URDF file
    # "joints" is the solution from your inverse kinematics solver
    transformation_matrixes = my_chain.forward_kinematics(joints, full_kinematics=True)
    
    # Get the nodes and the orientation from the transformation matrix
    for (index, link) in enumerate(my_chain.links):
        (node, orientation) = geometry.from_transformation_matrix(transformation_matrixes[index])
    
        # Add node corresponding to the link
        nodes.append(node)
    return nodes;

In [61]:
%matplotlib notebook
%matplotlib ipympl

from scipy.spatial import ConvexHull    
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

points = [[0.65612, 0.53440, 0.24175],
           [0.62279, 0.51946, 0.25744],
           [0.61216, 0.53959, 0.26394]]

p0, p1, p2 = points
x0, y0, z0 = p0
x1, y1, z1 = p1
x2, y2, z2 = p2

ux, uy, uz = u = [x1-x0, y1-y0, z1-z0]
vx, vy, vz = v = [x2-x0, y2-y0, z2-z0]

u_cross_v = [uy*vz-uz*vy, uz*vx-ux*vz, ux*vy-uy*vx]

point  = np.array(p0)
normal = np.array(u_cross_v)

d = -point.dot(normal)

xx, yy = np.meshgrid(range(10), range(10))

z = (-normal[0] * xx - normal[1] * yy - d) * 1. / normal[2]

target_position = [0,3,0]
my_chain.plot(my_chain.inverse_kinematics(target_position), ax, target=target_position)

for point in get_joints():
    v = point - point
    
    dot_product = np.dot(v,normal)
    print(dot_product > 0)

target_position = [0.1,0.2,0.03]
my_chain.plot(my_chain.inverse_kinematics(target_position), ax, target=target_position)

for point in get_joints():
    v = point - point
    
    dot_product = np.dot(v,normal)
    print(dot_product > 0)

target_position = [0.7,0.5,8]
my_chain.plot(my_chain.inverse_kinematics(target_position), ax, target=target_position)

for point in get_joints():
    v = point - point
    
    dot_product = np.dot(v,normal)
    print(dot_product > 0)

# plot the surface
plt3d = plt.figure().add_subplot(projection="3d")
plt3d.plot_surface(xx, yy, z)
plt.show()


hull = ConvexHull(reached)
fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot_trisurf(reached[:,0], reached[:,1], reached[:,2], triangles=hull.simplices, alpha=0.4)
fig, ax = plot_utils.init_3d_figure()

my_chain.plot(my_chain.inverse_kinematics(target_position), ax, target=target_position)


ValueError: shapes (4,) and (3,) not aligned: 4 (dim 0) != 3 (dim 0)